# runtime

> Everything that runs a model: native output capture, the context window, and the backend the harness talks to.

Three concerns that are really one. A model call goes out through `Backend`, the C++ engine underneath it writes to file descriptors nobody is watching, and the whole thing is bounded by a context window that has to be managed before it is exceeded rather than after.

In [ ]:
#| default_exp runtime

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail, expect_fail
from ramabana.core import resolve, ModelSpec

In [ ]:
#| export
import contextvars, copy, math, os, re, sys, threading, time
from contextlib import contextmanager
from dataclasses import dataclass
from fastcore.basics import patch
from ramabana.core import agent_err, env, force_tags, local_ctx, local_window, tool_channel

## Native output

A local engine is a C++ library that writes to file descriptors 1 and 2 directly. Its complaints never pass through Python and never reach a log handler. When a turn fails because the model was handed more tokens than it can hold, the only evidence is on a descriptor. This section reads that evidence and keeps the one line worth showing.

In [ ]:
#| export
MAX_KEEP = 8_000        # tail kept per call. An engine that logs a lot must not eat memory
_NOISE = ('created tensorflow lite', 'xnnpack delegate', 'metal delegate', 'tflite','loading model', 'initialized', 'gpu delegate', 'w0000', 'i0000')
_SIGNAL = ('error', 'fail', 'exceed', 'exceeds', 'too long', 'out of memory', 'oom','invalid', 'refus', 'cannot', 'unsupported', 'abort')

In [ ]:
#| export
def interesting(text, limit=4):
    "The lines of captured output a person should see: complaints, not chatter."
    from fastcore.basics import uniqueify
    out = []
    for ln in (text or '').splitlines():
        s = ln.strip()
        if not s: continue
        low = s.lower()
        if any(n in low for n in _NOISE): continue
        if any(g in low for g in _SIGNAL): out.append(s)
    return uniqueify(out)[-limit:]

An engine narrates its own startup on every call. The filter matches on words a complaint contains rather than on a log level. There isn't one.

In [ ]:
log = '''Created TensorFlow Lite XNNPACK delegate for CPU.
Loading model from /models/gemma-e4b
I0000 00:00:1730000000.000000 metal delegate ready
ERROR: input tokens 21014 exceeds max_num_tokens 16384
'''
interesting(log)

['ERROR: input tokens 21014 exceeds max_num_tokens 16384']

Duplicates collapse and only the most recent survive, because an engine that fails on every step of a tool loop repeats itself and the user needs one line, not forty.

In [ ]:
test_eq(interesting('ERROR: oom\nERROR: oom\nERROR: oom'), ['ERROR: oom'])
interesting('\n'.join(f'error {i}' for i in range(10)), limit=2)

['error 8', 'error 9']

`captured` tees a descriptor rather than swallowing it: the bytes still reach the terminal they were going to, and a buffer keeps a copy. It is serialised on a class lock, since two threads redirecting descriptor 2 at the same time would restore each other's copies and permanently detach the real one.

In [ ]:
#| export
class _Tee:
    "Copy one redirected descriptor to its original destination and an in-memory buffer."
    def __init__(self, fd):
        self.fd, self.buf, self.thread = fd, bytearray(), None
        self.saved = self.r = self.w = None
    def start(self):
        self.saved = os.dup(self.fd)              
        self.r, self.w = os.pipe()
        os.dup2(self.w, self.fd)
        os.close(self.w)
        self.w = None
        self.thread = threading.Thread(target=self._pump, daemon=True)
        self.thread.start()
    def _pump(self):
        while True:
            try: b = os.read(self.r, 4096)
            except OSError: break
            if not b: break
            self.buf += b
            del self.buf[:-MAX_KEEP]
            try: os.write(self.saved, b)              # still goes where it was going
            except OSError: pass
    def stop(self):
        # restore the descriptor before closing the pipe, so writes during teardown reach the original destination
        if self.saved is not None:
            try: os.dup2(self.saved, self.fd)
            except OSError: pass
        if self.r is not None:
            try: os.close(self.r)
            except OSError: pass
        if self.thread is not None: self.thread.join(timeout=1.0)
        if self.saved is not None:
            try: os.close(self.saved)
            except OSError: pass
        return self.buf.decode('utf-8', 'replace')

In [ ]:
#| export
class captured:
    "Context manager: `with captured() as cap: ...`, then read `cap.text`."
    _lock = threading.Lock()   # two threads redirecting one descriptor would restore each other's copies
    def __init__(self, fds=(1, 2), enabled=None):
        self.fds = fds
        self.text = ''
        # through `env`. The switch follows whatever prefix this application named
        self.enabled = ((env('NO_NATIVE_CAPTURE') or '').lower() not in ('1', 'true', 'yes')
                        if enabled is None else enabled)
        self._tees, self._held = [], False
    def __enter__(self):
        if not self.enabled: return self
        if not self._lock.acquire(timeout=0.5): return self
        self._held = True
        for fd in self.fds:
            t = _Tee(fd)
            try:
                sys.stdout.flush(); sys.stderr.flush()
                t.start()
                self._tees.append(t)
            except Exception: break
        return self
    def __exit__(self, *exc):
        parts = []
        for t in reversed(self._tees):
            try: parts.append(t.stop())
            except Exception: pass
        self._tees = []
        if self._held:
            self._held = False
            try: self._lock.release()
            except RuntimeError: pass
        self.text = ''.join(reversed(parts))
        return False

    @property
    def problems(self):
        "The captured lines worth reporting, as one string, or ''."
        return '\n'.join(interesting(self.text))

In [ ]:
#| export
def capture(fn, *a, **kw):
    "Call `fn`, returning `(result, captured_problem_text)`. Exceptions carry the text out too."
    cap, err, out = captured(), None, None
    with cap:
        try: out = fn(*a, **kw)
        except Exception as e: err = e
    if err is not None:   # after the block: the text does not exist until the pipe is drained
        err.native_output = cap.problems
        raise err
    return out, cap.problems

`capture` is the form callers use: it returns the result alongside whatever the engine complained about. A successful-but-suspicious call is still inspectable.

In [ ]:
def _engine_call():
    os.write(2, b'Created TensorFlow Lite XNNPACK delegate for CPU.\n')
    os.write(2, b'WARNING: input exceeds max_num_tokens 16384, truncating\n')
    return 'a short reply'

capture(_engine_call)

Created TensorFlow Lite XNNPACK delegate for CPU.


('a short reply', 'WARNING: input exceeds max_num_tokens 16384, truncating')

When the call raises instead, the text is attached to the exception as `native_output`. That is the reason for this module: the exception a Python caller sees is often `RuntimeError('')`, and the sentence explaining it was written to a descriptor before the stack unwound.

In [ ]:
def _engine_dies():
    os.write(2, b'ERROR: failed to allocate 4.00 GiB for KV cache\n')
    raise RuntimeError('')
try: capture(_engine_dies)
except RuntimeError as e: err = e
err.native_output

ERROR: failed to allocate 4.00 GiB for KV cache


'ERROR: failed to allocate 4.00 GiB for KV cache'

## Sizing a conversation

Every budget here is a fraction of the window rather than a constant, because the same code serves a 4k local model and a 200k cloud one. A fixed 16k reserve against a 4k window makes every turn "due for compaction", which is how an agent ends up compacting a two-message conversation forever.

In [ ]:
#| export
CHARS_PER_TOKEN = 3.25       #: ornith and qwen3 both measure 3.50. Estimate high: see `estimate_tokens`
RESERVE = 16_384             # headroom kept below the window: one full reply plus its tool results
KEEP_RECENT = 20_000         # tokens of recent conversation compaction does not touch
SUMMARY_PREFIX = 'Previous conversation summary:\n'
SURGICAL_POLICY = {'user': 2000, 'assistant': 150, 'call': 60, 'result': 35}

In [ ]:
#| export
def estimate_tokens(text, count=None):
    "Tokens in `text`: exact via `count` when a tokenizer is at hand, `CHARS_PER_TOKEN` otherwise."
    if not text: return 0
    if count is not None:
        try: return count(text)
        except Exception: pass
    return max(1, math.ceil(len(text) / CHARS_PER_TOKEN))

def halvings(budget, tries=3, floor=256):
    "A budget and its halvings, to retry a prompt whose fit was only estimated."
    if not budget: return [budget]
    return [budget] + [max(floor, budget >> i) for i in range(1, tries)]

def threshold(ctx, reserve=RESERVE):
    "The token count at which a conversation should be compacted, or None when there is no window."
    if not ctx or ctx <= 0: return None
    return max(1, ctx - min(reserve, max(1, ctx // 4)))   # a quarter of the window, for small models

def should_compact(used, ctx, reserve=RESERVE):
    "Whether `used` tokens against a `ctx` window has crossed the line."
    t = threshold(ctx, reserve)
    return bool(t and used >= t)

With no tokenizer at hand, tau's four-characters-per-token estimate is close enough to budget with. When a backend can count exactly, it passes its own counter in.

In [ ]:
estimate_tokens('the exporter writes one module per notebook'), estimate_tokens('', ), estimate_tokens('x' * 400)

(11, 0, 100)

`threshold` is where compaction becomes due. On a large window the full reserve applies. On a small one the quarter-window cap does, which is the difference between a usable local model and an unusable one.

In [ ]:
threshold(200_000), threshold(32_768), threshold(4096)

(183616, 24576, 3072)

In [ ]:
test_eq(threshold(4096), 3072)          # reserve capped at ctx//4, not 16k
test_eq(threshold(0), None)             # no window: never due
should_compact(3000, 4096), should_compact(3072, 4096)

(False, True)

## Reading a conversation

Two backends are in play and their message shapes differ. Aidialog objects with `content` parts, and provider dicts. Everything that inspects history goes through `_text`, `_role` and `_calls` so the difference is confined to three functions.

In [ ]:
#| export
def _text(m):
    "The readable text of a message in either backend's shape."
    if hasattr(m, 'content') and not isinstance(m, dict):        # aidialog Msg
        return '\n'.join(str(p.text) for p in m.content if getattr(p, 'text', None))
    if not isinstance(m, dict): return str(m)
    c = m.get('content', '')
    if isinstance(c, str): return c
    out = []
    for p in c or []:
        if not isinstance(p, dict): continue
        if p.get('type') == 'text': out.append(p.get('text', ''))
        elif p.get('type') == 'tool_response': out.append(f"[{p.get('name','tool')}] {p.get('response')}")
    return '\n'.join(x for x in out if x)

def _role(m): return getattr(m, 'role', None) or (m.get('role', '?') if isinstance(m, dict) else '?')

def _calls(m):
    "Tool call names on an assistant message, in either shape."
    tcs = getattr(m, 'tool_calls', None)
    if tcs is None and isinstance(m, dict): tcs = m.get('tool_calls')
    if not tcs:
        parts = getattr(m, 'content', None)
        if parts and not isinstance(m, dict):
            return [p.data.get('name', '?') for p in parts if getattr(p, 'type', '') == 'tool_use' and p.data]
        return []
    out = []
    for t in tcs:
        n = getattr(t, 'name', None) or (t.get('function', {}).get('name') if isinstance(t, dict) else None)
        if n: out.append(n)
    return out

A conversation for the examples below: a request, a tool call, its result, and a reply.

In [ ]:
hist = [
    {'role': 'user', 'content': 'the exporter duplicates cells, please fix it'},
    {'role': 'assistant', 'content': 'Reading the exporter.',
     'tool_calls': [{'function': {'name': 'view_file', 'arguments': {'path': 'nbdev/maker.py'}}}]},
    {'role': 'tool', 'content': 'def _make_exists(self, cells):\n    with self.fname.open("a") as f: ...'},
    {'role': 'assistant', 'content': 'It appends, so a second export duplicates every named cell.'},
]
[( _role(m), _text(m)[:40], _calls(m)) for m in hist]

[('user', 'the exporter duplicates cells, please fi', []),
 ('assistant', 'Reading the exporter.', ['view_file']),
 ('tool', 'def _make_exists(self, cells):\n    with ', []),
 ('assistant', 'It appends, so a second export duplicate', [])]

`serialise` renders history as the tagged block a summarizer reads, clipping tool results because they are almost all the bulk and almost none of the meaning.

In [ ]:
#| export
def serialise(msgs, mx=2000):
    "Messages as the tagged block the summarizer reads. Tool results clipped: they are the bulk."
    if not msgs: return '(no new messages)'
    out = []
    for i, m in enumerate(msgs, 1):
        out.append(f'<message index={i} role={_role(m)}>')
        if (t := _text(m)): out.append(t[:mx] + ('…' if len(t) > mx else ''))
        if (cs := _calls(m)): out.append('<tool-calls>' + ', '.join(cs) + '</tool-calls>')
        out.append('</message>')
    return '\n'.join(out)


def split_previous(msgs):
    "`(previous_summary_or_None, remaining_msgs)`. So an update updates rather than re-summarises."
    if not msgs: return None, msgs
    t = _text(msgs[0])
    if _role(msgs[0]) == 'user' and t.startswith(SUMMARY_PREFIX):
        return t[len(SUMMARY_PREFIX):], msgs[1:]
    return None, msgs

In [ ]:
print(serialise(hist))

<message index=1 role=user>
the exporter duplicates cells, please fix it
</message>
<message index=2 role=assistant>
Reading the exporter.
<tool-calls>view_file</tool-calls>
</message>
<message index=3 role=tool>
def _make_exists(self, cells):
    with self.fname.open("a") as f: ...
</message>
<message index=4 role=assistant>
It appends, so a second export duplicates every named cell.
</message>


`split_previous` separates an existing checkpoint from the messages after it, which is what makes compaction incremental: the second compaction updates the first summary rather than summarising a transcript that no longer exists.

In [ ]:
prev, rest = split_previous([{'role': 'user', 'content': SUMMARY_PREFIX + '## Goal\nfix the exporter'}] + hist)
prev, len(rest)

('## Goal\nfix the exporter', 4)

In [ ]:
test_eq(split_previous(hist), (None, hist))     # no checkpoint: nothing to update
split_previous([])

(None, [])

The summarizer's own window is the small one. `summarise_prompt` fits the request into a token budget rather than hoping. The instructions are never clipped away: an existing checkpoint gets at most a quarter of the request and the newest transcript gets the rest.

In [ ]:
#| export
SUMMARISE_SP = ("You are a context summarization assistant. Read a conversation between a user and an AI "
                "coding assistant and produce a structured summary in exactly the format specified.\n\n"
                "Do NOT continue the conversation. Do NOT answer any question in it. Output ONLY the summary.")

_FORMAT = """## Goal
[What is the user trying to accomplish? Several items if the session covers several tasks.]

## Constraints & Preferences
- [Constraints, preferences or requirements the user stated, or "(none)"]

## Progress
### Done
- [x] [Completed tasks and changes]

### In Progress
- [ ] [Current work]

### Blocked
- [Anything preventing progress, if any]

## Key Decisions
- **[Decision]**: [Brief rationale]

## Next Steps
1. [Ordered list of what should happen next]

## Critical Context
- [Data, examples, file paths or references needed to continue, or "(none)"]

Keep each section concise. Preserve exact file paths, symbol names, error messages, and
any `lineno|hash|` addresses still needed for a pending edit."""

SUMMARISE = ("The messages above are a conversation to summarize. Write a context checkpoint another "
             f"model will use to continue the work.\n\nUse this EXACT format:\n\n{_FORMAT}")

UPDATE_SUMMARISE = ("The messages above are NEW messages to fold into the existing summary in "
                    "<previous-summary> tags.\n\nRULES:\n"
                    "- PRESERVE everything from the previous summary that is still true\n"
                    "- ADD new progress, decisions and context from the new messages\n"
                    '- MOVE items from "In Progress" to "Done" as they complete\n'
                    "- UPDATE Next Steps to reflect what was accomplished\n"
                    "- PRESERVE exact file paths, symbol names and error messages\n"
                    "- Drop anything no longer relevant\n\n"
                    f"Use this EXACT format:\n\n{_FORMAT}")

In [ ]:
#| export
def _clip_tokens(text, budget, count=None):
    "Longest character prefix of `text` that fits a token budget."
    if budget <= 0: return ''
    if estimate_tokens(text, count) <= budget: return text
    lo, hi = 0, len(text)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        suffix = '…' if mid < len(text) else ''
        if estimate_tokens(text[:mid] + suffix, count) <= budget: lo = mid
        else: hi = mid - 1
    return text[:lo] + ('…' if lo < len(text) else '')


def summarise_prompt(msgs, extra='', max_tokens=None, count=None):
    "The bounded prompt handed to the summarizer, choosing fresh or update instructions."
    prev, rest = split_previous(msgs)
    base = UPDATE_SUMMARISE if prev is not None else SUMMARISE
    if extra.strip(): base = f'{base}\n\nAdditional focus: {extra.strip()}'
    transcript = serialise(rest)
    def build(body, old=prev):
        p = f'<conversation>\n{body}\n</conversation>\n\n'
        if old is not None: p += f'<previous-summary>\n{old}\n</previous-summary>\n\n'
        return p + base
    if not max_tokens or estimate_tokens(build(transcript), count) <= max_tokens: return build(transcript)
    old = prev
    if old is not None: old = _clip_tokens(old, max(64, max_tokens // 4), count)
    fixed = build('', old)
    room = max(0, max_tokens - estimate_tokens(fixed, count) - 4)
    body = _clip_tokens(transcript, room, count)
    out = build(body, old)
    while body and estimate_tokens(out, count) > max_tokens:
        body = body[:-1]
        out = build(body + '…', old)
    return _clip_tokens(out, max_tokens, count)

Unbounded, the prompt is the whole transcript plus the format instructions.

In [ ]:
p = summarise_prompt(hist)
test_eq(p.count('<message'), len(hist))
estimate_tokens(p)

324

Given a budget it comes in under it, every time, including budgets too small for the instructions themselves. A truncated instruction is better than a call the engine is guaranteed to refuse.

In [ ]:
[estimate_tokens(summarise_prompt(hist, max_tokens=n)) for n in (2000, 400, 120, 30)]

[324, 324, 120, 30]

In [ ]:
for n in (2000, 400, 120, 30): assert estimate_tokens(summarise_prompt(hist, max_tokens=n)) <= n
summarise_prompt(hist, max_tokens=30)

'<conversation>\n\n</conversation>\n\nThe messages above are a conversation to summarize. Write a context checkpoint another…'

`truncate_middle` keeps both ends of a value, since the informative parts of a path, an error or a diff are at its edges. `surgical_history` uses it to render old history as a compact DSL. The deterministic alternative to summarising, for when no summarizer model is available or its output cannot be trusted.

In [ ]:
#| export
def truncate_middle(text, budget, count=None, mark=' … '):
    "Keep both ends of text inside a token budget."
    text = str(text or '')
    if estimate_tokens(text, count) <= budget: return text
    if budget <= estimate_tokens(mark, count): return _clip_tokens(mark, budget, count)
    lo, hi, best = 0, len(text), mark
    while lo <= hi:
        n = (lo + hi) // 2
        left = (n + 1) // 2
        candidate = text[:left] + mark + (text[-(n-left):] if n-left else '')
        if estimate_tokens(candidate, count) <= budget: best, lo = candidate, n + 1
        else: hi = n - 1
    return best

In [ ]:
#| export
def _call_rows(m):
    "Canonical `(name, args)` calls from a backend assistant message."
    if not isinstance(m, dict): return []
    out = []
    for tc in m.get('tool_calls') or []:
        fn = tc.get('function') or {}
        out.append((fn.get('name', 'tool'), fn.get('arguments') or {}))
    content = m.get('content')
    for part in content if isinstance(content, list) else []:
        if isinstance(part, dict) and part.get('type') == 'tool_call':
            out.append((part.get('name', 'tool'), part.get('arguments') or {}))
    return out


def surgical_history(msgs, policy=None, count=None):
    "Render old history as a compact, readable DSL while preserving tool evidence."
    policy = {**SURGICAL_POLICY, **(policy or {})}
    rows = []
    for m in msgs:
        role, text = _role(m), _text(m).strip()
        if role == 'user' and text:
            rows.append('§ ' + truncate_middle(text, policy['user'], count) + ' §')
        elif role == 'assistant':
            if text: rows.append('» ' + truncate_middle(text, policy['assistant'], count) + ' »')
            for name, args in _call_rows(m):
                call = f"▶ {name}({', '.join(f'{k}={v!r}' for k,v in args.items())})"
                rows.append(truncate_middle(call, policy['call'], count))
        elif role == 'tool':
            result = ' ¶ '.join(x.strip() for x in text.splitlines() if x.strip())
            rows.append('> ' + truncate_middle(result, policy['result'], count))
    return '\n'.join(rows)

In [ ]:
truncate_middle('nbs/01_runtime.ipynb::Compactor.compact::summarise_prompt', 12)

'nbs/01_runtime.ipynb::C … pact::summarise_prompt'

Each role gets its own marker and its own budget. A reader (and a model) can still see the shape of what happened: request, call, evidence, conclusion.

In [ ]:
print(surgical_history(hist))

§ the exporter duplicates cells, please fix it §
» Reading the exporter. »
▶ view_file(path='nbdev/maker.py')
> def _make_exists(self, cells): ¶ with self.fname.open("a") as f: ...
» It appends, so a second export duplicates every named cell. »


The budgets are per role and overridable, because tool results are worth almost nothing once summarised while the user's own words are worth keeping nearly whole.

In [ ]:
print(surgical_history(hist, policy={'result': 8, 'assistant': 20}))

§ the exporter duplicates cells, please fix it §
» Reading the exporter. »
▶ view_file(path='nbdev/maker.py')
> def _make_exist … "a") as f: ...
» It appends, so a second export duplicates every named cell. »


## Reorienting the model

After its context is rewritten, a model is told what just happened. Every clause is here because leaving it out causes a specific failure: without the kernel sentence it re-imports and rebuilds data that is still live, without the skills sentence it works from a half-remembered skill, and without the last sentence it re-answers a question it already answered. Because a summary reads like an instruction to resume.

In [ ]:
#| export
def reorient(kernel_alive=True, skills=()):
    "What the model is told immediately after its context is rewritten."
    live = ("**Your context was rewritten to fit the window, but the kernel process was not touched.** "
            "The user's namespace, imports and variables are all still live exactly as they were -- do "
            "not re-import anything, do not rebuild data, and do not re-run setup. Call `list_vars` if "
            "you need to see what is there."
            if kernel_alive else
            "**Your context was rewritten and the kernel has restarted with a clean namespace.** "
            "Rebuild variables on demand; do not assume anything is still bound.")
    sk = (f"Skill text you read earlier is gone from your context; re-read it with `read_skill` before "
          f"relying on it ({', '.join(skills)})." if skills else
          "Any skill text you read earlier is gone from your context; re-read it before relying on it.")
    return (f'<system-reminder>\n{live}\n\n{sk}\n\n'
            'The summary above describes work in flight. If the last thing the user asked has already '
            'been answered and nothing is open, do not resume or re-answer anything -- reply with one '
            'short line and wait.\n</system-reminder>')


REORIENT = reorient()

In [ ]:
print(reorient(skills=('nbdev', 'fossick')))

<system-reminder>
**Your context was rewritten to fit the window, but the kernel process was not touched.** The user's namespace, imports and variables are all still live exactly as they were -- do not re-import anything, do not rebuild data, and do not re-run setup. Call `list_vars` if you need to see what is there.

Skill text you read earlier is gone from your context; re-read it with `read_skill` before relying on it (nbdev, fossick).

The summary above describes work in flight. If the last thing the user asked has already been answered and nothing is open, do not resume or re-answer anything -- reply with one short line and wait.
</system-reminder>


When the kernel did not survive, the promise changes, and it is the one part that must never be wrong.

In [ ]:
test_eq('do not re-import anything' in reorient(kernel_alive=False), False)
print(reorient(kernel_alive=False)[:180])

<system-reminder>
**Your context was rewritten and the kernel has restarted with a clean namespace.** Rebuild variables on demand; do not assume anything is still bound.

Any skill


## Notices on the way in

A submitted prompt earns notices before it reaches the model. These are cheap string checks, and each one exists because of a failure people actually hit. Most of all the approval notice: a person who types "go" after a long exchange is approving the thing under discussion, not the four other things the model listed on the way there.

In [ ]:
#| export
Q_NOTICE = ('This prompt ends with a question mark, so it is a question. Make only the tool calls needed '
            'to answer it, then answer it, then stop -- do not start the work it implies.')
READ_NOTICE = ('This prompt asks you to read something. Read the target in full now, before composing any '
               'response: a notebook with `notebook_cells` then `view_cell`, a file with `view_file`. '
               'Never answer from assumed or remembered contents.')
APPROVAL_NOTICE = ('This bare approval covers exactly what was explicitly agreed, and nothing more. Before '
                   'acting, check that each thing you are about to do was confirmed by the user -- not '
                   'merely proposed, listed or summarised by you. If approval of an item is uncertain, it '
                   'is not approved: ask.')
BTW_NOTICE = ('This prompt begins with "BTW" and is a side request. Answer it first, then resume the '
              'previous task if it has unfinished items. It does not cancel that task.')
ACTION_NOTICE = ('This is an action request, not a request for instructions or a plan. Use the available '
                 'execution/editing tools now, retry corrected calls when one fails, verify the requested '
                 'result exists, and only then answer with the completed result.')

_APPROVALS = ('go', 'ok', 'okay', 'yes', 'yep', 'sure', 'do it', 'go ahead', 'proceed')


def prompt_notices(prompt):
    "Notices a submitted prompt earns, from aai-coding's `UserPromptSubmit` hook."
    p = (prompt or '').strip()
    out = []
    if p.endswith('?'): out.append(Q_NOTICE)
    if re.search(r'\b(please read|read the|have a look at)\b', p.lower()): out.append(READ_NOTICE)
    if re.sub(r'^\W+|[\s.!]+$', '', p.lower()) in _APPROVALS: out.append(APPROVAL_NOTICE)
    if p.lower().startswith('btw'): out.append(BTW_NOTICE)
    low = p.lower()
    if (re.match(r'^(create|make|scale|run|execute|fix|change|add|remove|rename|convert|save)\b', low)
            or re.search(r'\bas\s+[a-zA-Z_]\w*\s*$', p)):
        out.append(ACTION_NOTICE)
    return out


def notices_block(prompt):
    "The notices for `prompt` as one reminder to append to it, or `''`."
    ns = prompt_notices(prompt)
    return '' if not ns else '\n\n<system-reminder>\n' + '\n\n'.join(ns) + '\n</system-reminder>'

#: What to say to a model that wrote a tool call as prose instead of emitting one. The tags channel
#: asks for exact punctuation, and a smaller model narrates the call about as often as it makes it.
TAG_REMINDER = ("That tool call arrived as prose rather than as a call, so nothing ran. Emit it "
                "again on its own: one <tool_call> block containing only JSON with `name` and "
                "`arguments`, and no other text in the message.")

A question earns the notice that says answer it and stop, rather than start the work it implies.

In [ ]:
test_eq(prompt_notices('can you fix the exporter?'), [Q_NOTICE])
prompt_notices('go')

['This bare approval covers exactly what was explicitly agreed, and nothing more. Before acting, check that each thing you are about to do was confirmed by the user -- not merely proposed, listed or summarised by you. If approval of an item is uncertain, it is not approved: ask.']

An imperative earns the opposite notice. Act, verify, then report. A prompt can earn more than one at once.

In [ ]:
[len(prompt_notices(p)) for p in ('fix the exporter', 'BTW please read nbs/01_runtime.ipynb', 'thanks')]

[1, 2, 0]

`notices_block` is the form the harness appends: one reminder, or nothing at all.

In [ ]:
print(notices_block('ok'))



<system-reminder>
This bare approval covers exactly what was explicitly agreed, and nothing more. Before acting, check that each thing you are about to do was confirmed by the user -- not merely proposed, listed or summarised by you. If approval of an item is uncertain, it is not approved: ask.
</system-reminder>


In [ ]:
test_eq(notices_block('thanks'), '')
notices_block('thanks')

''

## Notebook context, last mile

A tagged notebook is sent whole whenever it fits. Only when it does not does the user's per-cell policy take effect: cells marked `discard` go first, then the oldest `auto` cells, and a `keep` cell is never dropped. The `fits` callable belongs to the backend. The real window of the selected model decides. Not an estimate made here.

In [ ]:
#| export
def compact_notebook_context(prompt, fits):
    "Reduce a tagged notebook only when `prompt` does not fit. A `keep` cell is never removed."
    if not isinstance(prompt, str) or fits(prompt): return prompt
    match = re.search(r'<notebook(?P<attrs>[^>]*)>\n?(?P<body>.*?)\n?</notebook>', prompt, re.S)
    if not match: return prompt
    cells = list(re.finditer(r'<cell\b[^>]*\bcompact="(?P<mode>auto|keep|discard)"[^>]*>.*?</cell>',
                             match.group('body'), re.S))
    active = list(range(len(cells)))

    def rebuild(removed):
        omitted = [cells[n].group('mode') for n in sorted(removed)]
        body = '\n'.join(cells[n].group(0) for n in active if n not in removed)
        note = (f'<context-compacted omitted="{len(omitted)}" discard="{omitted.count("discard")}" '
                f'auto="{omitted.count("auto")}" />\n' if omitted else '')
        notebook = f'<notebook{match.group("attrs")}>\n{note}{body}\n</notebook>'
        return prompt[:match.start()] + notebook + prompt[match.end():]

    removed = set()
    for mode in ('discard', 'auto'):
        for n in active:
            if cells[n].group('mode') != mode: continue
            removed.add(n)
            candidate = rebuild(removed)
            if fits(candidate): return candidate
    return rebuild(removed)

A notebook of three cells, one of each policy:

In [ ]:
nb = '''<notebook path=analysis.ipynb>
<cell id=1 compact="discard">print(load_everything())  # 900 lines of output</cell>
<cell id=2 compact="auto">df = read_csv('sales.csv')</cell>
<cell id=3 compact="keep">model.fit(df)  # the cell the question is about</cell>
</notebook>

why did the fit fail?'''
print(compact_notebook_context(nb, fits=lambda s: True))

<notebook path=analysis.ipynb>
<cell id=1 compact="discard">print(load_everything())  # 900 lines of output</cell>
<cell id=2 compact="auto">df = read_csv('sales.csv')</cell>
<cell id=3 compact="keep">model.fit(df)  # the cell the question is about</cell>
</notebook>

why did the fit fail?


It fits. Nothing happened. Under a budget it fails, the `discard` cell goes first and the model is told, in the notebook, that something was removed.

In [ ]:
out = compact_notebook_context(nb, fits=lambda s: len(s) < 270)
print(out)

<notebook path=analysis.ipynb>
<context-compacted omitted="1" discard="1" auto="0" />
<cell id=2 compact="auto">df = read_csv('sales.csv')</cell>
<cell id=3 compact="keep">model.fit(df)  # the cell the question is about</cell>
</notebook>

why did the fit fail?


A tighter budget takes the oldest automatic cell as well. A `keep` cell is never dropped. Even when the result still does not fit, because that policy is the user's and the last mile is not where it gets overruled.

In [ ]:
tight = compact_notebook_context(nb, fits=lambda s: len(s) < 150)
test_eq('compact="keep"' in tight, True)
test_eq('load_everything' in tight, False)
print(tight)

<notebook path=analysis.ipynb>
<context-compacted omitted="2" discard="1" auto="1" />
<cell id=3 compact="keep">model.fit(df)  # the cell the question is about</cell>
</notebook>

why did the fit fail?


## The compactor

`Compactor` decides when to compact and does it. It is deliberately not a callback on either engine: compaction needs a second model call on a different, cheaper model, and then it has to replace the first model's history. Two things neither backend's callback system does. Keeping it out here also means the summary is available to write into the notebook, which is the point.

In [ ]:
#| export
class Compactor:
    "Decides when to compact, and does it. Not a callback on either engine."
    def __init__(self,
                 reserve=RESERVE,
                 keep_recent=KEEP_RECENT,
                 auto=True,                  # compact automatically on crossing the threshold
                 kernel_alive=True,          # what the reorientation note may promise
                 on_compact=None,            # called with the compacted checkpoint once it exists
                 strategy='summary'):         # 'summary' | 'surgical' deterministic DSL
        self.reserve, self.keep_recent, self.auto = reserve, keep_recent, auto
        self.strategy = strategy
        self.kernel_alive, self.on_compact = kernel_alive, on_compact
        self.count = 0
        self.last = ''
        self.note = 'not compacted'

    def due(self, backend):
        "Whether `backend` has crossed its threshold."
        return should_compact(backend.used_tokens, backend.spec.ctx, self.reserve)

    def budget(self, ctx=0, overhead=0):
        "How much recent conversation to keep, for a window of `ctx` holding `overhead` besides."
        if not ctx: return self.keep_recent
        return min(self.keep_recent, max(256, max(256, ctx - overhead) // 2))

    def overhead(self, backend, msgs, count=None):
        "What the window holds that is not this conversation, by subtraction from `used_tokens`."
        used = getattr(backend, 'used_tokens', 0) or 0
        if not used: return 0
        return max(0, used - sum(estimate_tokens(_text(m), count) + 8 for m in msgs))

    def _keep(self, msgs, count=None, ctx=0, overhead=0):
        "The tail to keep uncompacted, newest-first until the budget runs out. Whole messages only."
        sizes = [estimate_tokens(_text(m), count) + 8 for m in msgs]
        budget = self.budget(ctx, overhead)
        if sizes: budget = min(budget, max(256, sum(sizes)//2))
        kept, used = [], 0
        for m, n in zip(reversed(msgs), reversed(sizes)):
            if used + n > budget and kept: break
            kept.append(m); used += n
        kept.reverse()
        while kept and _role(kept[0]) != 'user': kept.pop(0)
        return kept

    def compact(self, backend, summariser, extra='', summary_ctx=0, summary_output=1024, summary_count=None):
        "Summarise `backend`'s conversation with `summariser(prompt, sp)` and replace it. The summary, or `''`."
        msgs = list(backend.hist or [])
        if not msgs:
            self.note = 'nothing to compact'
            return ''
        keep = self._keep(msgs, backend.count_tokens, getattr(backend.spec, 'ctx', 0),
                          self.overhead(backend, msgs, backend.count_tokens))
        older = msgs[:len(msgs) - len(keep)] if len(keep) < len(msgs) else msgs
        if not older:
            self.note = 'everything is recent; nothing to compact'
            return ''
        if self.strategy == 'surgical':
            text = surgical_history(older, count=backend.count_tokens)
            head = SUMMARY_PREFIX + text + '\n\n' + reorient(self.kernel_alive)
            try: backend.replace_hist(head, keep)
            except Exception as e:
                self.note = f'compaction checkpoint written but history not replaced ({agent_err(e)})'
                return text
            self.count += 1; self.last = text
            self.note = f'surgically compacted {len(older)} message(s), kept {len(keep)}'
            if self.on_compact:
                try: self.on_compact(text)
                except Exception: pass
            return text
        # the system prompt and the output share the window with this request
        input_budget = None
        if summary_ctx:
            sp_tokens = estimate_tokens(SUMMARISE_SP, summary_count)
            input_budget = max(128, summary_ctx - summary_output - sp_tokens - 64)
        # the budget is an estimate wherever the backend has no tokenizer, so a prompt built to
        # fit can still overflow
        text, err = '', None
        for b in halvings(input_budget):
            try:
                text = (summariser(summarise_prompt(older, extra, b, summary_count), SUMMARISE_SP) or '').strip()
                if text: break
            except Exception as e: err = e
        if not text:
            self.note = (f'compaction failed ({agent_err(err)})' if err is not None
                         else 'the summarizer returned nothing; conversation left alone')
            return ''
        head = SUMMARY_PREFIX + text + '\n\n' + reorient(self.kernel_alive)
        try: backend.replace_hist(head, keep)
        except Exception as e:
            self.note = f'compaction summary written but history not replaced ({agent_err(e)})'
            return text
        self.count += 1
        self.last = text
        self.note = f'compacted {len(older)} message(s), kept {len(keep)}'
        if self.on_compact:
            try: self.on_compact(text)
            except Exception: pass
        return text

`budget` is how much recent conversation survives, and it is capped at half the window for the same reason the reserve is. 20k of "recent" on a 4k model means the tail is the entire conversation, `older` is empty, and the compactor reports "nothing to compact" right up until the engine refuses the turn.

In [ ]:
c = Compactor()
c.budget(200_000), c.budget(32_768), c.budget(4096)

(20000, 16384, 2048)

In [ ]:
test_eq(c.budget(4096), 2048)
test_eq(c.note, 'not compacted')
c.strategy

'summary'

Compacting needs a live backend and a summarizer. The end-to-end example lives in [testing](04_testing.ipynb), where the doubles it needs are defined.

## Answers, without the thinking

A reasoning model emits `<think>...</think>` inline whenever the runtime does not separate it into a channel of its own. Every cheap job here reads a one-shot reply as data. A label, a summary, a completion to insert. The thinking has to come off before the caller sees it. A classifier that returns the model's deliberation returns nothing.

Some models never emit the opening tag at all, because their chat template writes it into the generation prompt and leaves the model to close it. That breaks any splitter looking for a `<think>` to begin with: the deliberation arrives as ordinary reply text, and only the closing tag comes back. It also happens once per *step*. A turn that calls tools re-enters the thought after every call. `answer_only` handles a finished reply, `ThinkFilter` handles a streamed one, and the cheap jobs sidestep it entirely by asking not to think.

In [ ]:
#| export
def answer_only(text):
    "A one-shot reply with the model's thinking removed, however the runtime left it."
    from urai import split_think
    out, _ = split_think(text or '')
    if '</think>' in out: out = out.partition('</think>')[2]
    if '<think>' in out: out = out.partition('<think>')[0]
    return out.strip()

The answer survives. The deliberation does not.

In [ ]:
answer_only('<think>7 times 6. Six sevens are 42.</think>\n\n42')

'42'

The shape that actually arrives from a template-primed reasoning model is the *unpaired* close: the prompt opened the block. The reply begins inside the thought and only the closing tag comes back.

In [ ]:
answer_only('Six sevens are 42. They asked for only the number.\n</think>\n\n42')

'42'

In [ ]:
test_eq(answer_only('<think>cut off at the cap and never closed'), '')
test_eq(answer_only('thinking, and then the cap hit\n'), 'thinking, and then the cap hit')
test_eq(answer_only('42'), '42')                       # a model with nothing to hide is untouched
answer_only('<think>a</think>yes')

'yes'

A template-primed model does this *once per step*, not once per turn. A turn that calls tools re-enters the thought after every call. `answer_only` reads a finished reply and can partition it. A stream has to be filtered as it arrives, and re-armed at each tool call.

In [ ]:
#| export
def prefills_think(chat):
    "Does this model's chat template open a `<think>` block and leave the model to close it?"
    tok = getattr(chat, 'tokenizer', None)
    if tok is None: return False
    try: p = tok.apply_chat_template([{'role': 'user', 'content': 'x'}], add_generation_prompt=True, tokenize=False)
    except Exception: return False
    return '<think>' in p and '</think>' not in p.rsplit('<think>', 1)[-1]

class ThinkFilter:
    "Drop a template-opened thinking block out of a raw chunk stream. A tool call re-arms it."
    TAG = '</think>'
    def __init__(self): self.thinking, self.buf, self.thought, self.answer = True, '', 0, 0
    def _tool(self, o):
        parts = o.get('content') or [] if isinstance(o, dict) else []
        return any(isinstance(p, dict) and p.get('type') == 'tool_call' for p in parts)
    def __call__(self, chunks):
        "Filter raw chunk dicts, yielding the same shape back."
        from urai import resp_text
        for o in chunks:
            if self._tool(o): self.thinking, self.buf = True, ''; yield o; continue
            if not self.thinking: self.answer += len(resp_text(o)); yield o; continue
            if not (txt := resp_text(o)): yield o; continue
            self.buf += txt; self.thought += len(txt)
            if (k := self.buf.find(self.TAG)) < 0: self.buf = self.buf[1 - len(self.TAG):]; continue
            self.thinking, out, self.buf = False, self.buf[k + len(self.TAG):].lstrip('\n'), ''
            if out: self.answer += len(out); yield {'content': [{'type': 'text', 'text': out}]}

The tags never survive a chunk boundary, and the answer arrives whole:

In [ ]:
def chunks(*texts): return [{'content': [{'type': 'text', 'text': t}]} for t in texts]
def texts(chunks):
    from urai import resp_text
    return ''.join(resp_text(c) for c in chunks)

f = ThinkFilter()
texts(f(chunks('Six sevens', ' are 42.\n</th', 'ink>\n\n', '42')))

'42'

In [ ]:
test_eq(texts(ThinkFilter()(chunks('deliberating', '</think>', 'yes'))), 'yes')

# a tool call re-arms it. The next step's thinking goes too
tool = {'content': [{'type': 'tool_call', 'name': 'search_code', 'arguments': {}}]}
out = list(ThinkFilter()(chunks('think 1', '</think>', 'calling ') + [tool] + chunks('think 2', '</think>', 'done')))
test_eq(texts(out), 'calling done')
test_eq(out[1], tool)                                  # the call itself is passed straight through

# thinking that never closed yields nothing, and says how much it swallowed
f = ThinkFilter()
test_eq(texts(f(chunks('cut off at the cap'))), '')
test_eq((f.thinking, f.thought, f.answer), (True, 18, 0))

# a closed thought and then the end of the turn: what `_stream` turns into a problem line
f = ThinkFilter()
test_eq(texts(f(chunks('deliberating', '</think>\n\n'))), '')
test_eq((f.thinking, f.answer), (False, 0))

## Usage

`Usage` is one turn's token accounting, and it adds. A session total is `sum(usages)` and the same object renders the status bar.

In [ ]:
#| export
MAX_STEPS = 40
ONESHOT_TOKENS = 1024     # a cheap job's default output cap

@dataclass
class Usage:
    model:str=''; input:int=0; output:int=0; total:int=0; cached:int=0
    cache_write:int=0; reasoning:int=0; cost:float=0.; turns:int=0
    def __add__(self,o):
        if o is None: return self
        fs=('input','output','total','cached','cache_write','reasoning','cost','turns')
        return Usage(model=o.model or self.model,**{k:getattr(self,k)+getattr(o,k) for k in fs})
    def __radd__(self,o): return self if o in (None,0) else self+o
    def __sub__(self,o):
        "What this counter has added since `o`. A backend counts cumulatively. A turn is a delta."
        if o is None: return self
        fs=('input','output','total','cached','cache_write','reasoning','cost','turns')
        return Usage(model=self.model or o.model, **{k:max(0, getattr(self,k)-getattr(o,k)) for k in fs})
    def __repr__(self):
        p=[f'{self.total:,} tok',f'in {self.input:,}',f'out {self.output:,}']
        if self.cached:p.append(f'cached {100*self.cached/max(self.input,1):.0f}%')
        if self.reasoning:p.append(f'thought {self.reasoning:,}')
        if self.cost:p.append(f'${self.cost:.4f}')
        if self.model:p.append(self.model.split('/')[-1])
        return ' · '.join(p)
    def dict(self): return dict(self.__dict__)

In [ ]:
a = Usage(model='anthropic/claude-sonnet-5', input=12_000, output=800, total=12_800, cached=9_000, cost=0.041, turns=1)
a

12,800 tok · in 12,000 · out 800 · cached 75% · $0.0410 · claude-sonnet-5

Addition keeps the model name of the right-hand side, since a session that switched models should report the one in use now.

In [ ]:
b = Usage(model='openai/gpt-5.6-terra', input=3_000, output=200, total=3_200, cost=0.008, turns=1)
sum([a, b], None)

16,000 tok · in 15,000 · out 1,000 · cached 60% · $0.0490 · gpt-5.6-terra

In [ ]:
test_eq((a + None).total, a.total)      # a turn that produced no usage changes nothing
test_eq(sum([a, b], None).turns, 2)
a.dict()['cached']

9000

## Backend

`Backend` is the harness's whole view of a model: start it, send to it, stream from it, count its tokens, replace its history. Nothing above this class knows whether the model is a local engine or an HTTP API.

The contract that matters is that it does not raise. A model that cannot start, a turn that fails mid-stream and an engine that returns nothing all produce a readable note and a recorded problem, because the alternative is a traceback in a chat window.

In [ ]:
#| export
IMG_TOKENS = 1024

#: What a built OpenAI content part calls a picture and a sound, so one that has already been
#: through `mk_oai_content` is charged as media rather than stringified base64.
_MEDIA_PARTS = ('image_url', 'input_audio')

def _parts(msg):
    "An outgoing message as its text and its media count."
    ps = list(msg) if isinstance(msg,(list,tuple)) else [msg]
    txt,n = [],0
    for p in ps:
        if isinstance(p,(bytes,bytearray,os.PathLike)): n += 1
        elif isinstance(p,dict) and p.get('type') in _MEDIA_PARTS: n += 1
        elif isinstance(p,dict) and p.get('type') == 'text': txt.append(str(p.get('text','')))
        else: txt.append(str(p))
    return '\n'.join(txt), n

In [ ]:
#| export
class Backend:
    kind='?'
    _tag_reminded=False
    def __init__(self,spec,sp='',tools=(),approve=None,tool_max_len=None,shared=False,**kw):
        self.spec,self.sp,self.tools,self.approve=spec,sp,list(tools),approve
        self.tool_max_len,self.shared,self.kw=tool_max_len,shared,kw
        self.chat,self.use,self.note=None,Usage(model=spec.model_id),'not started'
        self.problems,self.last_native,self._tried,self.run=[], '', False, None
        self._resume_hist=None
        self._used=0                 # last occupancy the engine reported. See `used_tokens`
        self.lock=threading.Lock()
    @property
    def ready(self): return self.chat is not None
    @property
    def busy(self): return self.lock.locked()
    @property
    def hist(self): return getattr(self.chat,'hist',[]) if self.chat else []
    def problem(self,text):
        text=(text or '').strip()
        if text and (not self.problems or self.problems[-1]!=text): self.problems.append(text)
        del self.problems[:-20]
        return text
    def _failed(self,what,e):
        native=getattr(e,'native_output','') or ''
        self.note=f'{self.spec.name} {what} ({agent_err(e)})'+(f' -- {native}' if native else '')
        return self.problem(self.note)
    def start(self):
        if self._tried:return self.chat
        self._tried=True
        try:
            self.chat=self._start()
            if self._resume_hist is not None:
                self.restore_hist(self._resume_hist); self._resume_hist=None
            self.note=f'{len(self.tools)} tools'
        except Exception as e:self.chat=None; self._failed('unavailable',e)
        return self.chat
    def retry(self): self._tried,self.chat=False,None; return self.start()
    def set_approve(self,approve):
        self.approve=approve
        if self.chat:self.chat.approve=approve
        return self
    def refresh(self,sp,tools):
        self.sp,self.tools=sp,list(tools)
        if self.chat:self._refresh(); self.note=f'{len(self.tools)} tools'
        return self
    def close(self):
        if aux:=getattr(self,'_oneshot_chat',None):
            try:aux.close()
            except Exception:pass
            self._oneshot_chat=None
        if self.chat:
            try:self.chat.close()
            except Exception:pass
            self.chat=None
    def cancel(self):
        "Ask this backend's chat to stop the turn in flight."
        if not self.chat:return False
        f=getattr(self.chat,'cancel',None)
        if f is None:
            self.problem(f'{self.spec.name} cannot be stopped mid-turn')
            return False
        try:f()
        except Exception as e:self._failed('could not be stopped',e); return False
        return True
    def _sync_callbacks(self):
        "Give the live chat every registered callback it is missing. Callers hold `lock`."
        add = getattr(self.chat, 'add_cb', None)
        if add is None: return
        held = getattr(self.chat, 'cbs', ())
        for cb in getattr(self, '_callbacks', ()):
            if not any(isinstance(x, cb) for x in held): add(cb)
    def send(self,msg,run=None,**kw):
        if self.start() is None:return self.note
        with self.lock:
            self.run=run
            self._tag_reminded=False   # one reminder per turn. See `TAG_REMINDER`
            self._sync_callbacks()     # anything registered while a turn was running
            try:
                for again in (True,False):
                    try:
                        out=self._send(msg,**kw)
                        if run is not None and run.cancelled:return ''
                        self.use=self._usage(); self._check_reply(out)
                        # one corrective turn, appended rather than a re-run: the narrated call is
                        # already said, and asking again is the only way to still get the call
                        if not self._tag_reminded and self._needs_tag_retry(out):
                            self._tag_reminded=True
                            out=self._send(TAG_REMINDER,**kw); self.use=self._usage()
                        return out or self._empty()
                    except Exception as e:
                        if run is not None and run.cancelled:return ''
                        if not (again and self._recover(e)):return self._failed('failed',e)
            finally:self.run=None
    def stream(self,msg,run=None,**kw):
        if self.start() is None:yield self.note; return
        with self.lock:
            self.run=run
            self._tag_reminded=False   # one reminder per turn. See `TAG_REMINDER`
            self._sync_callbacks()     # anything registered while a turn was running
            try:
                for again in (True,False):
                    n,buf=0,[]
                    try:
                        for c in self._stream(msg,**kw):
                            if run is not None and run.cancelled:return
                            n+=len(c or ''); buf.append(c or ''); yield c
                        if run is not None and run.cancelled:return
                        self.use=self._usage()
                        self._check_reply(''.join(buf))
                        if not self._tag_reminded and self._needs_tag_retry(''.join(buf)):
                            self._tag_reminded=True
                            yield '\n\n'
                            for c2 in self._stream(TAG_REMINDER,**kw):
                                if run is not None and run.cancelled:return
                                n+=len(c2 or ''); yield c2
                            self.use=self._usage()
                        if not n:yield self._empty(True)
                        return
                    except Exception as e:
                        if run is not None and run.cancelled:return
                        # only before anything reached the screen: a retry cannot unsay a chunk
                        if not (again and not n and self._recover(e)):
                            yield f'\n\n{self._failed("failed",e)}'; return
            finally:self.run=None
    def _recover(self,e):
        "Fix what made `e` happen, if this backend knows how. One retry is worth taking. No by default."
        return False
    def _check_reply(self,text):
        "Look at a finished reply for a failure the transport could not raise. Nothing by default."
        return text
    def _needs_tag_retry(self,text):
        "Whether this reply narrated a tool call instead of emitting one. No by default."
        return False
    def _empty(self,strict=False):
        why=f'{self.spec.name} returned nothing'+(f' -- {self.last_native}' if self.last_native else '')
        return self.problem(why) if strict else (f'({why})' if self.last_native else '(no reply)')
    
    def oneshot(self,prompt,sp='',max_tokens=None):
        if self.start() is None or not self.lock.acquire(False):return ''
        try:return answer_only(self._oneshot(prompt,sp,max_tokens) or '')
        except Exception as e:self._failed('one-shot failed',e); return ''
        finally:self.lock.release()
    
    def spawn(self,sp='',tools=(),**kw): raise NotImplementedError
    def replace_hist(self,summary,keep=()):
        if not self.chat:raise RuntimeError('nothing to compact: the model is not running')
        self._replace_hist(summary,list(keep)); return self
    
    def snapshot_hist(self):
        "A detached model-history checkpoint suitable for an in-process branch."
        return copy.deepcopy(list(self.hist))
    
    def resume_hist(self,hist):
        "Restore canonical history now, or after this backend starts lazily."
        if self.chat:return self.restore_hist(hist)
        self._resume_hist=copy.deepcopy(list(hist or [])); return self
    
    def restore_hist(self,hist):
        "Restore a checkpoint and rebuild provider conversation state."
        if not self.chat:raise RuntimeError('nothing to restore: the model is not running')
        self.chat.hist[:]=copy.deepcopy(list(hist or []))
        recreate=getattr(self.chat,'_recreate_conv',None)
        if recreate:recreate()
        return self
    
    def revise_last_assistant(self,text):
        "Replace the last prose assistant message in a restored checkpoint."
        if not self.chat:raise RuntimeError('nothing to revise: the model is not running')
        msg=next((m for m in reversed(self.chat.hist) if isinstance(m,dict) and m.get('role')=='assistant' and not m.get('tool_calls')),None)
        if msg is None:raise ValueError('checkpoint has no assistant response')
        msg['content']=str(text)
        msg.pop('channels',None); msg.pop('usage',None)
        recreate=getattr(self.chat,'_recreate_conv',None)
        if recreate:recreate()
        return self
    
    def count_tokens(self,text):
        try:return self.chat.count_tokens(text or '') if self.chat else estimate_tokens(text)
        except Exception:return estimate_tokens(text)
    
    @property
    def used_tokens(self):
        "used tokens or token count from rishi chat."
        try:
            if self.chat: self._used = self.chat.token_count
        except Exception: pass
        return self._used
    
    @property
    def pct_full(self):return self.used_tokens/max(self.spec.ctx,1)
    def pending_tokens(self,msg):
        "Tokens the pending message adds. Media is charged `IMG_TOKENS`, never its repr."
        try:
            if self.chat and hasattr(self.chat,'render'):return self.count_tokens(self.chat.render(msg))
        except Exception:pass
        text,media = _parts(msg)
        return self.count_tokens(text)+8+IMG_TOKENS*media
    
    def projected_tokens(self,msg):return self.used_tokens+self.pending_tokens(msg)
    def fits(self,msg,reserve=None):
        if not self.spec.ctx:return True
        reserve=min(1024,max(128,self.spec.ctx//4)) if reserve is None else max(0,reserve)
        return self.projected_tokens(msg)<=max(1,self.spec.ctx-reserve)
    
    def _start(self):raise NotImplementedError
    def _send(self,msg,**kw):raise NotImplementedError
    def _stream(self,msg,**kw):raise NotImplementedError
    def _oneshot(self,prompt,sp,max_tokens):raise NotImplementedError
    def _replace_hist(self,summary,keep):raise NotImplementedError
    def _usage(self):return self.use
    def _refresh(self):raise NotImplementedError

`Backend` itself implements none of the engine hooks, which makes it a good demonstration of the failure path: starting it fails, and the failure is reported rather than raised.

In [ ]:
b = Backend(resolve('gemma-e4b'))
b.start(), b.note

(None, 'gemma-e4b unavailable (NotImplementedError: )')

A send against a backend that never started returns that same note. A caller that ignores the distinction still shows the user something true.

In [ ]:
test_eq(b.send('hello'), b.note)
b.problems

['gemma-e4b unavailable (NotImplementedError: )']

A picture in a pending message is priced at `IMG_TOKENS` rather than tokenized. Its bytes carry no text to count, and `str` on them gives a repr that prices one screenshot at hundreds of thousands of tokens -- enough to fail a turn that would have fit, and to spend a compaction on the way there.

In [ ]:
test_eq(b.pending_tokens([bytes(120_000), 'what is this?']),
        b.pending_tokens('what is this?') + IMG_TOKENS)

Repeated problems are recorded once and the list is bounded, since an engine that fails on every step of a tool loop would otherwise fill memory with the same sentence.

In [ ]:
for _ in range(30): b.problem('ERROR: oom')
len(b.problems), b.problems[-1]

(2, 'ERROR: oom')

Token counting falls back to the estimate when no conversation exists yet. Budgeting works before the first call as well as after it.

In [ ]:
b.count_tokens('a prompt of some length'), b.used_tokens, round(b.pct_full, 3)

(6, 0, 0.0)

`fits` is what `compact_notebook_context` is handed. It answers against the real window of this spec, keeping a reserve for the reply.

In [ ]:
big = 'x' * 200_000
b.fits('a short prompt'), b.fits(big)

(True, False)

In [ ]:
test_eq(Backend(ModelSpec('unbounded', 'remote', 'v/m', ctx=0)).fits(big), True)   # no window, no limit
b.spec.ctx

16384

A one-shot answer arrives without the thinking, wherever the engine put it.

In [ ]:
class Thinker(Backend):
    def _start(self): return self
    def _oneshot(self, prompt, sp, max_tokens): return 'They want one label.\n</think>\n\nsuccess'

test_eq(Thinker(resolve('gemma-e4b')).oneshot('the tests pass'), 'success')
Thinker(resolve('gemma-e4b')).oneshot('the tests pass')

'success'

History operations refuse rather than corrupt: there is nothing to compact, restore or revise until a model is actually running.

In [ ]:
test_fail(lambda: b.replace_hist('summary'), contains='not running')
test_fail(lambda: b.restore_hist([]), contains='not running')
test_fail(lambda: b.revise_last_assistant('...'), contains='not running')

## RishiBackend

Rishi runs every model, local or hosted. There is one subclass. It translates a `ModelSpec` into rishi's `Chat`, adapts per-runtime quirks (litert wants its token ceiling and constrained decoding. MLX wants a separate completion-only conversation so a suggestion never sees prior suggestions), and converts rishi's usage into `Usage`.

LiteRT models run on the GPU without being asked for it: rishi requests that backend by default and warns its way down to the CPU on a machine whose build has no delegate. `RAMABANA_LITERT_BACKEND` is the way to *overrule* that. `cpu` pins a model to the CPU, `gpu` asks for what rishi would have asked for anyway. An explicit `backend=` (or `eng_kw={'backend': ...}`) still takes precedence over both. `RISHI_LITERT_GPU=0` turns the default off at the layer that owns it.

In [ ]:
#| export
_MK_CHAT = None
@contextmanager
def use_chat(f):
    "Build model conversations with `f` for the duration, instead of rishi's `Chat`."
    global _MK_CHAT   # process-global, which is why this is a block and not a setting
    old, _MK_CHAT = _MK_CHAT, f
    try: yield f
    finally: _MK_CHAT = old

class RishiBackend(Backend):
    kind='rishi'
    def __init__(self,*a,max_steps=MAX_STEPS,**kw):
        self.max_steps,self._prefill=max_steps,None; super().__init__(*a,**kw)
    @property
    def prefilled_think(self):
        "Whether this model's template opens a thinking block the model has to close."
        if self._prefill is None:self._prefill=prefills_think(self.chat)
        return self._prefill
    @property
    def tool_channel(self):
        "Where this backend's tool schemas actually travel. The chat answers once there is one."
        return tool_channel(self.spec,self.chat)
    def _runtime_kw(self):
        import os
        kw={**getattr(self.spec, 'config', {}), **self.kw}
        if key_env := kw.pop('api_key_env', None): kw['api_key'] = os.environ.get(key_env)
        if self.spec.runtime in ('remote','copilot') and tool_channel(self.spec)=='tags': kw.setdefault('tool_mode','tags')
        if self.spec.runtime in ('llama','ollama'): kw.setdefault('n_ctx',self.spec.ctx)
        if self.spec.runtime=='litert':
            eng=dict(kw.pop('eng_kw',{}) or {})
            if 'backend' not in eng and 'backend' not in kw and (backend := env('LITERT_BACKEND')):
                from litert_lm import Backend as LB
                backends = {'cpu': LB.CPU, 'gpu': LB.GPU}
                if backend.lower() not in backends: raise ValueError(f'unknown LiteRT backend {backend!r}; use cpu or gpu')
                kw['backend'] = backends[backend.lower()]()
            eng.setdefault('max_num_tokens',self.spec.ctx)
            kw['eng_kw']=eng
            if conv := dict(kw.pop('conv_kw',{}) or {}): kw['conv_kw']=conv   # rishi constrains a tool call itself
        return kw
    def _measure(self):
        "Narrow the window to what the model was trained for, capped by what is worth filling."
        if not self.spec.local: return
        if real := local_window(self.spec.runtime,self.spec.model_id):
            from dataclasses import replace
            self.spec=replace(self.spec,ctx=min(real,local_ctx(self.spec.name)))
    def _start(self):
        from rishi import Chat
        self._measure()
        return self._fitted((_MK_CHAT or Chat)(self.spec.model_id,runtime=self.spec.runtime,sp=self.sp,
                    tools=self.tools,approve=self.approve,tool_max_len=self.tool_max_len,
                    max_steps=self.max_steps,ctx_limit=self.spec.ctx,**self._runtime_kw()))
    def _fitted(self,chat):
        "Take the window from the engine, which is the only thing that knows it."
        try: real=int(chat.engine.n_ctx())
        except Exception: return chat
        if real and real<self.spec.ctx:
            from dataclasses import replace
            self.spec=replace(self.spec,ctx=real); chat.ctx_limit=real
        return chat
    def spawn(self,sp='',tools=(),**kw):
        if self.start() is None:raise RuntimeError(self.note)
        shared={'engine':self.chat.engine} if self.spec.local and hasattr(self.chat,'engine') else {}
        return type(self)(self.spec,sp=sp,tools=tools,tool_max_len=self.tool_max_len,
                          max_steps=self.max_steps,shared=True,**shared,**kw)
    def _turn_kw(self, kw):
        "Apply hosted turn controls at the layer Rishi owns. Chat.__call__ only accepts generation controls."
        kw = dict(kw or {})
        effort = kw.pop('reasoning_effort', None)
        if effort is not None and hasattr(self.chat, 'reasoning_effort'):
            self.chat.reasoning_effort = effort
        return kw
    def _send(self,msg,**kw):
        from urai import resp_text
        return answer_only(resp_text(self.chat(msg,**self._turn_kw(kw))))
    MCP_REFUSED=('mcp','strict_mcp_config','allowed_tools','disallowed','not permitted','policy')
    def _recover(self,e):
        "Learn that this model's wire tool channel is closed. Later turns stop trying it."
        if not self.tools or tool_channel(self.spec,self.chat)=='tags':return False
        if not any(s in f'{e}'.lower() for s in self.MCP_REFUSED):return False
        force_tags(self.spec.model_id,agent_err(e))
        self.problem(f'{self.spec.name}: the wire refused the tool schemas, so they now travel in '
                     f'the system prompt instead ({agent_err(e)})')
        hist=self.hist
        self.close(); self.retry()
        if self.chat is None:return False
        if hist:self.restore_hist(hist)
        return True
    def _check_reply(self,text):
        "Report a tag call that came back as prose, which is what the tags channel costs."
        if self._needs_tag_retry(text):
            how=('a <tool_call> block came back as prose' if '<tool_call' in (text or '')
                 else 'a tool call came back as bare JSON, with no <tool_call> tags around it')
            self.problem(f'{self.spec.name}: {how} rather than as a call, so this model is not '
                         'punctuating the tags channel reliably')
        return text
    def _needs_tag_retry(self,text):
        """A reply on the tags channel that shows a call it never made.

        The shape is rishi's to know, beside the parser that reads it. Only the tool names are
        ours: a reply naming something this backend does not carry is prose about JSON.
        """
        from urai import tag_call_shape
        if tool_channel(self.spec,self.chat)!='tags': return False
        return tag_call_shape(text,[getattr(t,'__name__','') for t in self.tools])
    def _stream(self,msg,**kw):
        kw=self._turn_kw(kw)
        if not self.prefilled_think:yield from self.chat(msg,stream=True,**kw); return
        from urai import StreamFormatter
        f=ThinkFilter()
        yield from StreamFormatter().format_stream(f(self.chat(msg,stream='raw',**kw)))
        if f.thought and not f.answer:
            self.problem(f'{self.spec.name} spent the whole turn thinking ({f.thought} characters) '
                         'and never answered; route `turn` to a larger model, or raise its output cap')
    def _oneshot(self,prompt,sp,max_tokens):
        if self.spec.runtime!='mlx':
            return self.chat.oneshot(prompt,sp,think=False,max_tokens=max_tokens or ONESHOT_TOKENS)
        if getattr(self,'_oneshot_chat',None) is None:
            if not hasattr(self.chat,'engine'):   # nothing to share: a replayed chat has no engine
                return self.chat.oneshot(prompt,sp,think=False,max_tokens=max_tokens or ONESHOT_TOKENS)
            from rishi import Chat
            self._oneshot_chat=Chat(self.spec.model_id,runtime='mlx',engine=self.chat.engine,think=False,
                                    sp=sp,ctx_limit=self.spec.ctx,max_output_tokens=ONESHOT_TOKENS)
        c=self._oneshot_chat
        c.sp=sp
        c.hist[:]=[c.mk_msg(prompt)]
        from urai import resp_text
        return resp_text(c._model_step(max_tokens or ONESHOT_TOKENS))
    def _replace_hist(self,summary,keep):
        if not hasattr(self.chat,'_recreate_conv'):
            raise RuntimeError(f'{type(self.chat).__name__} cannot have its history replaced')
        self.chat.hist[:]=self.chat.mk_msgs([summary,*keep]); self.chat._recreate_conv()
    def _usage(self):
        # a chat with no counter at all, rather than one that spent nothing
        if (u:=getattr(self.chat,'use',None)) is None: return Usage(model=self.spec.model_id)
        return Usage(model=u.model or self.spec.model_id,input=u.prompt_tokens,output=u.completion_tokens,
                     total=u.total_tokens,cached=u.cached_tokens,cost=u.cost,turns=u.n)
    def _refresh(self): self.chat.reconfigure(sp=self.sp,tools=self.tools)

# Dead names from when llama.cpp and FastLLM were separate backends.
def __getattr__(name):
    if name in ('LlamaBackend','FastllmBackend'):
        import warnings
        warnings.warn(f'{name} is a deprecated alias for RishiBackend; model execution goes '
                      'through Rishi for every runtime. Use RishiBackend.',
                      DeprecationWarning, stacklevel=2)
        return RishiBackend
    raise AttributeError(f'module {__name__!r} has no attribute {name!r}')

def make_backend(spec,**kw):return RishiBackend(spec,**kw)

`use_chat` is the seam a recording goes into. Rishi's `CachedChat` is what a docs page puts there. A real model's answers replay with no weights and no network. Here it is a chat that answers from a script, which is the same contract with nothing to download.

In [ ]:
from ramabana.core import ModelSpec

built = []
class StubChat:
    def __init__(self, model, **kw): built.append((model, kw)); self.hist = []
    def oneshot(self, prompt, sp='', think=None, max_tokens=None): return 'replayed'
    def close(self): pass

with use_chat(StubChat):
    be = make_backend(ModelSpec('stub', 'remote', 'openai/gpt-x', 8000))
    assert be.start() is not None
    test_eq(built[0][0], 'openai/gpt-x')            # the seam built the conversation, not rishi
    test_eq(be.oneshot('a cheap job'), 'replayed')
    test_eq(be._usage().total, 0)                   # a replay metered nothing, and says so
test_eq(_MK_CHAT, None)                             # process-global, and scoped to its block

In [ ]:
# rishi decides this now: constrained decoding is on when a litert chat has tools. Ramabana
# does not name it. What matters here is that nothing of ours is left setting the old flag.
def _a_tool(query: str) -> str:
    "A tool."
    return ''
kw = make_backend(resolve('gemma-e4b'), tools=[_a_tool])._runtime_kw()
assert 'enable_constrained_decoding' not in kw.get('conv_kw', {})
assert 'constrained_decoding_config' not in kw.get('conv_kw', {})
mine = dict(constrained_decoding_config='mine')
test_eq(make_backend(resolve('gemma-e4b'), tools=[_a_tool], conv_kw=mine)._runtime_kw()['conv_kw'], mine)

`make_backend` is the constructor callers use, and the older names remain as aliases because model execution now always goes through rishi.

In [ ]:
from litert_lm import Backend as LB
r = make_backend(resolve('gemma-e4b'), eng_kw=dict(backend=LB.GPU()))
# The dead names still resolve, and say so on the way past.
import warnings
from ramabana import runtime
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    assert runtime.LlamaBackend is runtime.FastllmBackend is runtime.RishiBackend
    assert [x.category for x in w] == [DeprecationWarning, DeprecationWarning]
r.kind, type(r).__name__

('rishi', 'RishiBackend', True)

A litert spec is handed its context window as the engine's token ceiling, and asks for constrained decoding only when there are tools to constrain to.

In [ ]:
r._runtime_kw()

{'eng_kw': {'backend': GPU(gpu_decode_steps_per_sync=None),
  'max_num_tokens': 16384}}

In [ ]:
r2 = make_backend(resolve('gemma-e4b'), tools=[interesting])
test_eq(r2._runtime_kw().get('conv_kw'), None)      # rishi names it, not us
r2._runtime_kw()['eng_kw']

In [ ]:
# The accelerator goes to rishi as `backend=`, the parameter it builds the engine from.
import os
os.environ['RAMABANA_LITERT_BACKEND'] = 'gpu'
try:
    k = make_backend(resolve('gemma-e4b'))._runtime_kw()
    test_eq((type(k['backend']), 'backend' in k['eng_kw']), (type(LB.GPU()), False))
    k2 = make_backend(resolve('gemma-e4b'), eng_kw=dict(backend=LB.CPU()))._runtime_kw()
    test_eq((type(k2['eng_kw']['backend']), 'backend' in k2), (type(LB.CPU()), False))  # left alone
finally: del os.environ['RAMABANA_LITERT_BACKEND']

Nothing above has started an engine: `_start` is the first thing that touches rishi, and it is called on first send. Building a backend is free.

In [ ]:
r.ready, r.busy, r.note

(False, False, 'not started')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

In [ ]:
#| export
@dataclass
class Run:
    "A foreground or delegated model call with bounded cancellation."
    id: str
    kind: str = 'root'
    question: str = ''
    model: str = ''
    parent: object = None
    grace: float = 0.25
    state: str = 'pending'
    started: float = 0.
    ended: float = 0.
    backend: object = None

    def __post_init__(self):
        import threading
        self.children, self._lock, self._done = [], threading.RLock(), threading.Event()
        if self.parent is not None: self.parent.children.append(self)

    @property
    def terminal(self): return self.state in ('completed', 'cancelled', 'detached', 'terminated', 'failed')
    @property
    def cancelled(self): return self.state in ('cancelling', 'cancelled', 'detached', 'terminated')

    def child(self, question='', model=''):
        import uuid
        return Run(f'run_{uuid.uuid4().hex[:12]}', 'child', question, model, self, self.grace)

    def start(self, backend=None):
        import time
        with self._lock:
            if self.state != 'pending': return False
            self.state, self.backend, self.started = 'running', backend, time.time()
            return True

    def attach(self, backend):
        with self._lock:
            self.backend = backend
            cancelled = self.cancelled
        if cancelled and backend is not None:
            threading.Thread(target=lambda: backend.cancel(), daemon=True).start()
        return not cancelled

    def finish(self, state='completed'):
        import time
        with self._lock:
            if self.terminal: return self
            self.state = 'cancelled' if self.cancelled else state
            self.ended = time.time(); self._done.set()
        return self

    def _mark_cancel(self):
        "Mark this run and every descendant cancelled, and return the backends left to stop."
        # Marking is the whole pass and stopping is the pass after: stopping a backend releases the worker
        # blocked on it, which takes the next queued child at once -- so marking and stopping together let a
        # released worker start a sibling, and a cancelled run went on spawning what it was cancelled to stop.
        with self._lock:
            if self.terminal: return []
            # a pending run has nothing of its own to stop, but what it started still does
            pending = self.state == 'pending'
            self.state = 'cancelling'
            children, backend = list(self.children), None if pending else self.backend
        if pending: self.finish('cancelled')
        out = [] if backend is None else [backend]
        for child in children: out += child._mark_cancel()
        return out

    def request_cancel(self):
        for backend in self._mark_cancel():
            threading.Thread(target=lambda b=backend: b.cancel(), daemon=True).start()
        return self

    def terminate(self):
        with self._lock:
            if self.state in ('completed', 'cancelled', 'terminated', 'failed'): return self
            children, backend = list(self.children), self.backend
        for child in children: child.terminate()
        if backend is not None:
            f = getattr(backend, 'terminate', None) or getattr(backend, 'close', None)
            if f is not None: threading.Thread(target=f, daemon=True).start()
        with self._lock:
            self.state, self.ended = 'terminated', time.time(); self._done.set()
        return self

    def wait(self, grace=None):
        import time
        end = time.monotonic() + (self.grace if grace is None else max(0, grace))
        for child in list(self.children):
            left = max(0, end - time.monotonic())
            child._done.wait(left)
            if not child.terminal: child.detach()
        left = max(0, end - time.monotonic())
        self._done.wait(left)
        if not self.terminal: self.detach()
        return self

    def detach(self):
        import time
        with self._lock:
            if self.terminal: return self
            self.state, self.ended = 'detached', time.time(); self._done.set()
        for child in list(self.children):
            if not child.terminal: child.detach()
        return self

    def cancel(self, grace=None): return self.request_cancel().wait(grace)

    def dict(self):
        return {'id': self.id, 'parent_id': getattr(self.parent, 'id', None), 'kind': self.kind,
                'question': self.question, 'model': self.model, 'state': self.state,
                'started': self.started, 'ended': self.ended,
                'children': [child.dict() for child in self.children]}


A run that cannot stop its provider becomes `detached` after the grace period. Detachment is terminal for the caller. Late provider output is not part of the run.


In [ ]:
import threading, time

class _Blocking:
    def __init__(self, stoppable=True): self.stoppable, self.released = stoppable, threading.Event()
    def cancel(self):
        if self.stoppable: self.released.set()

r = Run('run_cancelled', grace=.05)
r.start(_Blocking())
threading.Thread(target=lambda: (r.backend.released.wait(), r.finish()), daemon=True).start()
test_eq(r.cancel().state, 'cancelled')

r = Run('run_detached', grace=.01)
r.start(_Blocking(False))
test_eq(r.cancel().state, 'detached')


In [ ]:
#| export
_current_run = contextvars.ContextVar('ramabana_run', default=None)

def current_run(): return _current_run.get()

@contextmanager
def run_context(run):
    token = _current_run.set(run)
    try: yield run
    finally: _current_run.reset(token)


A cancelled backend does not retry and does not emit chunks produced after cancellation.


In [ ]:
class _CancelStream(Backend):
    def __init__(self):
        from ramabana.core import ModelSpec
        super().__init__(ModelSpec('cancel-test', 'fake', 'fake/cancel', ctx=1000))
        self.first, self.release, self.calls, self.recoveries = threading.Event(), threading.Event(), 0, 0
    def _start(self): return self
    def _stream(self, msg, **kw):
        self.calls += 1
        yield 'first'
        self.first.set(); self.release.wait()
        yield 'late'
    def cancel(self): self.release.set(); return True
    def _usage(self): return Usage()
    def _recover(self, e): self.recoveries += 1; return True

be, run, chunks = _CancelStream(), Run('run_late', grace=.05), []
run.start(be)
t = threading.Thread(target=lambda: chunks.extend(be.stream('x', run=run)), daemon=True)
t.start(); assert be.first.wait(.2)
run.cancel(); t.join(.2)
test_eq(chunks, ['first'])
test_eq((be.calls, be.recoveries), (1, 0))


In [ ]:
class _Closable:
    def __init__(self): self.closed = threading.Event()
    def close(self): self.closed.set()

r = Run('run_terminated', grace=0)
r.start(_Closable()); r.detach(); r.terminate()
test_eq(r.state, 'terminated')
assert r.backend.closed.wait(.2)


In [ ]:
#| export
@patch
def add_cb(self:Backend, cb):
    """Register one Rishi callback class, for this chat and for any that replaces it.

    A turn holds `lock` for its whole length and Rishi walks `chat.cbs` while it runs, so a caller
    on another thread records the callback and the running turn takes it up at its own boundary.
    Splicing into that list from outside can drop or repeat a callback in the turn already going.
    """
    callbacks = getattr(self, '_callbacks', [])
    if cb not in callbacks: callbacks.append(cb)
    self._callbacks = callbacks
    if self.lock.acquire(False):
        try: self._sync_callbacks()
        finally: self.lock.release()
    return self

@patch
def _start_callbacks(self:Backend): self._sync_callbacks()

@patch
def start(self:Backend):
    if self._tried: return self.chat
    self._tried = True
    try:
        self.chat = self._start()
        self._start_callbacks()
        if self._resume_hist is not None:
            self.restore_hist(self._resume_hist); self._resume_hist = None
        self.note = f'{len(self.tools)} tools'
    except Exception as e: self.chat = None; self._failed('unavailable', e)
    return self.chat

In [ ]:
#| export
from urai import ChatCallback
class TokenLogger(ChatCallback):
    "Report Rishi's provider-reported usage after each response."
    order = 20
    sink = None
    def after_response(self):
        u = self.chat.use
        cost = u.cost if isinstance(u.cost, (int, float)) else 0.   # a provider may report null
        (type(self).sink or print)(
            f'[rishi usage] model={u.model or "?"} input={u.prompt_tokens} '
            f'output={u.completion_tokens} total={u.total_tokens} cached={u.cached_tokens} '
            f'cost=${cost:.6f}')

CHAT_CALLBACKS = {'token_logger': TokenLogger}